# Waste-bin-Map
**Name:** Raushan Vivek  |  **Roll no.:** CS25M115  
**Assignment:** *Explore the mapped waste-bin observations, inspect their details, and view the associated field photographs.*  
**Wesite Link:** https://raushanvivek9.github.io/uisa_cs25m115/index.html
**Question 1:** https://raushanvivek9.github.io/uisa_cs25m115/Question1/index.html



In [14]:
# Interactive Web Mapping of Waste-Bin Observations

import geopandas as gpd
import folium
import pandas as pd
import os
import html

# ============================================================
# 1. FILE PATH

GPKG_FILE = "cs25m115.gpkg"


In [15]:
# 2. CHECK AVAILABLE LAYERS
print("Available layers:")
print(gpd.list_layers(GPKG_FILE))


Available layers:
                name geometry_type
0        notes_point       Point Z
1         notes_line  LineString Z
2      notes_polygon     Polygon Z
3  notes_attachments           NaN


In [16]:

# 3. READ WASTE-BIN POINT OBSERVATIONS

bins = gpd.read_file(
    GPKG_FILE,
    layer="notes_point"
)

print("\nWaste-bin observations:")
print(bins.head())

print("\nWaste-bin columns:")
print(list(bins.columns))

print("\nData types:")
print(bins.dtypes)



Waste-bin observations:
                                     uuid    color title note  \
0  {fb7f2cf4-d821-4885-be45-0e82f7b7ca68}  #4daf4a   WB1   40   
1  {3470a67b-649c-439b-b95c-07fa1a9b84cd}  #377eb8   WB2   10   
2  {ff8cddf2-c087-4a44-ab7c-b993df7e43dd}  #377eb8   WB3   30   
3  {705aca39-81b1-48eb-9476-5a2bce76b8ac}  #4daf4a   WB4   70   
4  {493e138e-0a4f-47c5-bb75-6eab5c3d750b}  #377eb8   WB5   60   

                         timestamp                               geometry  
0 2026-09-05 13:16:49.189000+00:00    POINT Z (79.59447 13.7061 96.25938)  
1 2026-09-05 13:21:10.292000+00:00    POINT Z (79.59493 13.70661 101.519)  
2 2026-09-05 13:28:14.534000+00:00   POINT Z (79.59489 13.70829 97.01942)  
3 2026-09-05 13:31:04.544000+00:00  POINT Z (79.59437 13.70768 103.84358)  
4 2026-09-05 13:33:19.016000+00:00   POINT Z (79.59454 13.70716 98.78376)  

Waste-bin columns:
['uuid', 'color', 'title', 'note', 'timestamp', 'geometry']

Data types:
uuid                         str
co

In [17]:
# 4. READ ATTACHMENTS

attachments = gpd.read_file(
 GPKG_FILE,
 layer="notes_attachments"
)

print("\nAttachment data:")
print(attachments.head())

print("\nAttachment columns:")
print(list(attachments.columns))

print("\nAttachment data types:")
print(attachments.dtypes)



Attachment data:
                                          note_layer  \
0  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
1  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
2  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
3  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
4  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   

                                note_uuid                            media  \
0  {effee446-2a77-492f-806d-edeb1536c144}  DCIM/JPEG_20260905131541433.jpg   
1  {fb7f2cf4-d821-4885-be45-0e82f7b7ca68}  DCIM/JPEG_20260905131707224.jpg   
2  {3470a67b-649c-439b-b95c-07fa1a9b84cd}  DCIM/JPEG_20260905132124992.jpg   
3  {ff8cddf2-c087-4a44-ab7c-b993df7e43dd}  DCIM/JPEG_20260905132844458.jpg   
4  {705aca39-81b1-48eb-9476-5a2bce76b8ac}  DCIM/JPEG_20260905133123925.jpg   

  description                        timestamp  
0        None 2026-09-05 13:15:41.434000+00:00  
1        None 2026-09-05 13:17:07.225000+00:00  
2        None 2026-09-05 13:2

In [21]:
# 2. Convert to EPSG:4326 for Folium

if bins.crs != "EPSG:4326":
    bins = bins.to_crs(epsg=4326)

In [22]:
# 3. Convert timestamp to string
bins["timestamp"] = bins["timestamp"].astype(str)

In [23]:
# 4. Create map
# ------------------------------------------------------------

center_lat = bins.geometry.y.mean()
center_lon = bins.geometry.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    control_scale=True
)

In [24]:
# 5. Create layer
waste_bins = folium.FeatureGroup(
    name="Waste Bin Observations"
)

waste_bins.add_to(m)

In [31]:
# 6. Add waste-bin points
# ------------------------------------------------------------

for _, row in bins.iterrows():

    lat = row.geometry.y
    lon = row.geometry.x

    popup_html = f"""
    <div style="width:300px">

        <h4>Waste Bin Observation</h4>

        <b>Student:</b> CS25M115_Raushan Vivek<br><br>

        <b>Title:</b> {html.escape(str(row["title"]))}<br><br>

        <b>Field Note:</b> {html.escape(str(row["note"]))}<br><br>

        <b>Timestamp:</b> {html.escape(str(row["timestamp"]))}<br><br>

        <b>Observation ID:</b> {html.escape(str(row["uuid"]))}<br><br>

        <b>Location:</b>
        {lat:.6f}, {lon:.6f}

    </div>
    """

    popup = folium.Popup(
        popup_html,
        max_width=400
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=7,
        fill=True,
        fill_opacity=0.8,
        popup=popup,
        tooltip=row["title"]
    ).add_to(waste_bins)

In [32]:
# 7. Add Layer Control
# ------------------------------------------------------------

folium.LayerControl().add_to(m)

In [33]:
# 8. Save map
# ------------------------------------------------------------

m.save("waste_bin_map.html")

# Display map
m

In [34]:
attachments = gpd.read_file(
    "cs25m115.gpkg",
    layer="notes_attachments"
)

print(attachments.columns)
print(attachments.head())

Index(['note_layer', 'note_uuid', 'media', 'description', 'timestamp'], dtype='str')
                                          note_layer  \
0  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
1  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
2  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
3  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   
4  Notes___Point_275e38ab_95de_41a4_980d_aa83e272...   

                                note_uuid                            media  \
0  {effee446-2a77-492f-806d-edeb1536c144}  DCIM/JPEG_20260905131541433.jpg   
1  {fb7f2cf4-d821-4885-be45-0e82f7b7ca68}  DCIM/JPEG_20260905131707224.jpg   
2  {3470a67b-649c-439b-b95c-07fa1a9b84cd}  DCIM/JPEG_20260905132124992.jpg   
3  {ff8cddf2-c087-4a44-ab7c-b993df7e43dd}  DCIM/JPEG_20260905132844458.jpg   
4  {705aca39-81b1-48eb-9476-5a2bce76b8ac}  DCIM/JPEG_20260905133123925.jpg   

  description                        timestamp  
0        None 2026-09-05 13:15:41.434000+00:00  
1        No

In [37]:
import geopandas as gpd
import folium
import pandas as pd
import os
import html

# ============================================================
# 1. Read waste-bin observations
# ============================================================

bins = gpd.read_file(
    "cs25m115.gpkg",
    layer="notes_point"
)

# ============================================================
# 2. Read photographs / attachments
# ============================================================

attachments = gpd.read_file(
    "cs25m115.gpkg",
    layer="notes_attachments"
)

# ============================================================
# 3. Convert waste-bin points to EPSG:4326
# ============================================================

if bins.crs != "EPSG:4326":
    bins = bins.to_crs(epsg=4326)

# ============================================================
# 4. Convert timestamp to string
# ============================================================

bins["timestamp"] = bins["timestamp"].astype(str)

# ============================================================
# 5. Join photograph information to waste-bin observations
# ============================================================

attachments = attachments[
    ["note_uuid", "media"]
].copy()

bins = bins.merge(
    attachments,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

# ============================================================
# 6. Create map
# ============================================================

center_lat = bins.geometry.y.mean()
center_lon = bins.geometry.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    control_scale=True
)

# ============================================================
# 7. Create waste-bin layer
# ============================================================

waste_bins = folium.FeatureGroup(
    name="Waste Bin Observations"
)

waste_bins.add_to(m)

# ============================================================
# 8. Add each waste-bin observation
# ============================================================

for _, row in bins.iterrows():

    lat = row.geometry.y
    lon = row.geometry.x

    title = html.escape(str(row["title"]))
    note = html.escape(str(row["note"]))
    timestamp = html.escape(str(row["timestamp"]))
    uuid = html.escape(str(row["uuid"]))

    # Photo filename
    photo = row["media"]

    # --------------------------------------------------------
    # Photo HTML
    # --------------------------------------------------------

    if pd.notna(photo):

        photo = str(photo)

        photo_html = f"""
        <br>
        <b>Photograph:</b><br>
        <img src="{photo}"
             width="280"
             style="margin-top:8px;">
        """

    else:

        photo_html = "<br><b>Photograph:</b> Not available"


    # --------------------------------------------------------
    # Popup
    # --------------------------------------------------------

    popup_html = f"""
    <div style="width:320px">

        <h4>Waste Bin Observation</h4>

        <b>Student:</b> CS25M115_Raushan Vivek
        <br><br>

        <b>Title:</b>
        {title}
        <br><br>

        <b>Field Note:</b>
        {note}
        <br><br>

        <b>Timestamp:</b>
        {timestamp}
        <br><br>

        <b>Observation ID:</b>
        {uuid}
        <br><br>

        <b>Coordinates:</b>
        {lat:.6f}, {lon:.6f}

        {photo_html}

    </div>
    """

    popup = folium.Popup(
        popup_html,
        max_width=400
    )

    # --------------------------------------------------------
    # Marker
    # --------------------------------------------------------

    folium.CircleMarker(
        location=[lat, lon],
        radius=7,
        fill=True,
        fill_opacity=0.8,
        popup=popup,
        tooltip=title
    ).add_to(waste_bins)

# ============================================================
# 9. Layer Control
# ============================================================

folium.LayerControl().add_to(m)

# ============================================================
# 10. Save HTML
# ============================================================

m.save("index.html")

# Display map
m